In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🏛 Civic Complaint Classification — ML Analysis\n",
    "### CSE Data Science Project\n",
    "**Model:** TF-IDF + Logistic Regression  \n",
    "**Categories:** Road, Garbage, Water, Electricity  \n",
    "**Language:** English + Hinglish"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Imports ──────────────────────────────────────────────\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.patches as mpatches\n",
    "import seaborn as sns\n",
    "import numpy as np\n",
    "from sklearn.feature_extraction.text import TfidfVectorizer\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.naive_bayes import MultinomialNB\n",
    "from sklearn.model_selection import train_test_split, cross_val_score\n",
    "from sklearn.metrics import (\n",
    "    classification_report, confusion_matrix,\n",
    "    accuracy_score, ConfusionMatrixDisplay\n",
    ")\n",
    "from wordcloud import WordCloud\n",
    "\n",
    "# Style\n",
    "plt.rcParams['figure.facecolor'] = '#0D0D0F'\n",
    "plt.rcParams['axes.facecolor']   = '#16161A'\n",
    "plt.rcParams['text.color']       = '#F0EEE8'\n",
    "plt.rcParams['axes.labelcolor']  = '#F0EEE8'\n",
    "plt.rcParams['xtick.color']      = '#7A7880'\n",
    "plt.rcParams['ytick.color']      = '#7A7880'\n",
    "plt.rcParams['axes.edgecolor']   = '#2A2A32'\n",
    "plt.rcParams['grid.color']       = '#2A2A32'\n",
    "plt.rcParams['font.family']      = 'monospace'\n",
    "\n",
    "COLORS = ['#F5C842', '#6BCF7F', '#42B4F5', '#F57C42']\n",
    "print('✅ Libraries loaded!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Dataset Preparation"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "train_texts = [\n",
    "    # ROAD (30)\n",
    "    'road is broken','pothole on the street','road needs repair',\n",
    "    'street light not working','road damaged after rain','highway has big holes',\n",
    "    'footpath is broken','speed breaker is damaged','road is flooded',\n",
    "    'traffic signal not working','bridge is damaged','no streetlight on highway',\n",
    "    'road marking is not visible','flyover has cracks','road is slippery',\n",
    "    'sadak toot gayi hai','sadak pe gadda ho gaya hai','road bahut kharab hai',\n",
    "    'sadak ki repair nahi hui','street light band hai','sadak pe bada gaddha hai',\n",
    "    'footpath toot gaya','highway pe bahut holes hai','sadak pe paani jama hai',\n",
    "    'signal kaam nahi kar raha','sadak ka repair kab hoga',\n",
    "    'road ke khadde se accident ho gaya','pulia toot gayi hai',\n",
    "    'speed breaker bahut bada hai','sadak pe marking nahi hai',\n",
    "\n",
    "    # GARBAGE (30)\n",
    "    'garbage not collected','trash is overflowing','waste lying on street',\n",
    "    'dustbin is full','garbage smell is very bad','no garbage pickup since days',\n",
    "    'waste not being removed','garbage truck has not come','litter all over the place',\n",
    "    'open garbage dump near house','stray dogs near garbage pile','garbage blocking the road',\n",
    "    'garbage burning in open','plastic waste on footpath','dead animal on road not removed',\n",
    "    'kachra nahi utha','kachra bahut zyada jama ho gaya hai','dustbin full ho gaya hai',\n",
    "    'safai nahi ho rahi','kachra gadi nahi aayi','kachra sadak pe pada hai',\n",
    "    'safai wala nahi aaya','ganda kachra jama ho gaya','badboo aa rahi hai kachra se',\n",
    "    'nagar palika kachra nahi utha rahi','khula kachraghara ghar ke paas hai',\n",
    "    'kachra jal raha hai khule mein','bazar mein bahut gandagi hai',\n",
    "    'dustbin hi nahi hai hamare area mein','drain kachra se bhar gayi hai',\n",
    "\n",
    "    # WATER (30)\n",
    "    'no water supply','water pipe is leaking','dirty water coming from tap',\n",
    "    'water shortage in area','water tank is empty','sewage water on road',\n",
    "    'water pressure is very low','borewell is not working','no water for 3 days',\n",
    "    'muddy water from tap','water meter is broken','pipeline burst on main road',\n",
    "    'water not reaching upper floors','sewage mixing with drinking water',\n",
    "    'water smells bad from tap','paani nahi aa raha','nal se paani band hai',\n",
    "    'paani ki pipe toot gayi','ganda paani aa raha hai','paani ka pressure bahut kam hai',\n",
    "    'paani tank khali hai','2 din se paani nahi aaya','naali ka paani sadak pe aa gaya',\n",
    "    'paani ka rang peela hai','boring kaam nahi kar rahi',\n",
    "    'paani mein badboo aa rahi hai','3 din se paani nahi hai',\n",
    "    'pipe se paani waste ho raha hai','tanker bhi nahi aa raha',\n",
    "    'paani peene layak nahi hai',\n",
    "\n",
    "    # ELECTRICITY (30)\n",
    "    'no electricity since morning','power cut for 3 hours','transformer is burnt',\n",
    "    'power outage in colony','electric wire hanging low on road',\n",
    "    'meter reading is wrong','no power supply since 2 days',\n",
    "    'electricity comes and goes every hour','no electricity in the whole street',\n",
    "    'electricity department not responding','electric pole is leaning and dangerous',\n",
    "    'fuse blown in our area','electric sparks from wire near house',\n",
    "    'load shedding every day','inverter not charging due to low voltage',\n",
    "    'bijli nahi aa rahi','light chali gayi subah se',\n",
    "    'bijli ka bill bahut zyada aaya hai','transformer jal gaya hai',\n",
    "    'bijli baar baar aa jaati hai','wire sadak pe latkaa hua hai',\n",
    "    'bijli vibhag ka koi jawab nahi','meter galat reading de raha hai',\n",
    "    '2 din se bijli nahi hai','bijli ke khambe jhuk gaya hai',\n",
    "    'hamare area mein andhera hai raat ko','load shedding roz ho rahi hai',\n",
    "    'bijli pole sadak pe gir gaya','taar toota pada hai koi nahi uthata',\n",
    "    'emergency mein bijli gul ho gayi',\n",
    "]\n",
    "\n",
    "train_labels = ['road']*30 + ['garbage']*30 + ['water']*30 + ['electricity']*30\n",
    "\n",
    "print(f'✅ Total samples: {len(train_texts)}')\n",
    "print(f'   Road: 30 | Garbage: 30 | Water: 30 | Electricity: 30')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. Dataset Distribution"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))\n",
    "fig.suptitle('Dataset Distribution', color='#F0EEE8', fontsize=14, fontweight='bold')\n",
    "\n",
    "categories = ['Road', 'Garbage', 'Water', 'Electricity']\n",
    "counts = [30, 30, 30, 30]\n",
    "\n",
    "# Bar chart\n",
    "bars = ax1.bar(categories, counts, color=COLORS, width=0.5, edgecolor='none')\n",
    "ax1.set_title('Samples per Category', color='#F0EEE8')\n",
    "ax1.set_ylabel('Count', color='#F0EEE8')\n",
    "ax1.set_ylim(0, 40)\n",
    "ax1.grid(axis='y', alpha=0.3)\n",
    "for bar, count in zip(bars, counts):\n",
    "    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,\n",
    "             str(count), ha='center', va='bottom', color='#F0EEE8', fontweight='bold')\n",
    "\n",
    "# Pie chart\n",
    "wedges, texts, autotexts = ax2.pie(\n",
    "    counts, labels=categories, colors=COLORS,\n",
    "    autopct='%1.0f%%', startangle=90,\n",
    "    wedgeprops={'edgecolor': '#0D0D0F', 'linewidth': 2}\n",
    ")\n",
    "for t in texts: t.set_color('#F0EEE8')\n",
    "for at in autotexts: at.set_color('#0D0D0F'); at.set_fontweight('bold')\n",
    "ax2.set_title('Category Split', color='#F0EEE8')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('dataset_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0D0D0F')\n",
    "plt.show()\n",
    "print('✅ Saved: dataset_distribution.png')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Model Training + Accuracy"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train/Test Split\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    train_texts, train_labels, test_size=0.2, random_state=42, stratify=train_labels\n",
    ")\n",
    "\n",
    "# TF-IDF\n",
    "vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1)\n",
    "X_train_vec = vectorizer.fit_transform(X_train)\n",
    "X_test_vec  = vectorizer.transform(X_test)\n",
    "\n",
    "# Logistic Regression\n",
    "lr_model = LogisticRegression(max_iter=1000)\n",
    "lr_model.fit(X_train_vec, y_train)\n",
    "lr_pred = lr_model.predict(X_test_vec)\n",
    "lr_acc  = accuracy_score(y_test, lr_pred)\n",
    "\n",
    "# Naive Bayes\n",
    "nb_model = MultinomialNB()\n",
    "nb_model.fit(X_train_vec, y_train)\n",
    "nb_pred = nb_model.predict(X_test_vec)\n",
    "nb_acc  = accuracy_score(y_test, nb_pred)\n",
    "\n",
    "print('=' * 45)\n",
    "print(f'  Logistic Regression Accuracy : {lr_acc*100:.2f}%')\n",
    "print(f'  Naive Bayes Accuracy         : {nb_acc*100:.2f}%')\n",
    "print('=' * 45)\n",
    "print(f'  Train size : {len(X_train)}')\n",
    "print(f'  Test size  : {len(X_test)}')\n",
    "print('=' * 45)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Confusion Matrix"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "fig.suptitle('Confusion Matrix Comparison', color='#F0EEE8', fontsize=14, fontweight='bold')\n",
    "\n",
    "labels = ['road', 'garbage', 'water', 'electricity']\n",
    "\n",
    "for ax, pred, title in zip(\n",
    "    axes,\n",
    "    [lr_pred, nb_pred],\n",
    "    ['Logistic Regression', 'Naive Bayes']\n",
    "):\n",
    "    cm = confusion_matrix(y_test, pred, labels=labels)\n",
    "    sns.heatmap(\n",
    "        cm, annot=True, fmt='d', ax=ax,\n",
    "        xticklabels=labels, yticklabels=labels,\n",
    "        cmap='YlOrRd', linewidths=0.5,\n",
    "        linecolor='#0D0D0F',\n",
    "        cbar_kws={'shrink': 0.8}\n",
    "    )\n",
    "    ax.set_title(title, color='#F0EEE8', fontweight='bold')\n",
    "    ax.set_xlabel('Predicted', color='#F0EEE8')\n",
    "    ax.set_ylabel('Actual', color='#F0EEE8')\n",
    "    ax.tick_params(colors='#7A7880')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0D0D0F')\n",
    "plt.show()\n",
    "print('✅ Saved: confusion_matrix.png')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 5. Model Comparison"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, ax = plt.subplots(figsize=(8, 5))\n",
    "\n",
    "models = ['Logistic\\nRegression', 'Naive\\nBayes']\n",
    "accs   = [lr_acc * 100, nb_acc * 100]\n",
    "colors = ['#C8F04A', '#42B4F5']\n",
    "\n",
    "bars = ax.bar(models, accs, color=colors, width=0.4, edgecolor='none')\n",
    "ax.set_ylim(0, 110)\n",
    "ax.set_ylabel('Accuracy (%)', color='#F0EEE8')\n",
    "ax.set_title('Model Accuracy Comparison', color='#F0EEE8', fontweight='bold')\n",
    "ax.axhline(y=100, color='#2A2A32', linestyle='--', alpha=0.5)\n",
    "ax.grid(axis='y', alpha=0.2)\n",
    "\n",
    "for bar, acc in zip(bars, accs):\n",
    "    ax.text(\n",
    "        bar.get_x() + bar.get_width()/2,\n",
    "        bar.get_height() + 1,\n",
    "        f'{acc:.1f}%',\n",
    "        ha='center', va='bottom',\n",
    "        color='#F0EEE8', fontweight='bold', fontsize=13\n",
    "    )\n",
    "\n",
    "winner = 'Logistic Regression' if lr_acc >= nb_acc else 'Naive Bayes'\n",
    "ax.text(0.5, 0.05, f'🏆 Winner: {winner}',\n",
    "        transform=ax.transAxes, ha='center',\n",
    "        color='#C8F04A', fontsize=11)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0D0D0F')\n",
    "plt.show()\n",
    "print('✅ Saved: model_comparison.png')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 6. Classification Report"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('LOGISTIC REGRESSION')\n",
    "print('=' * 55)\n",
    "print(classification_report(y_test, lr_pred, target_names=labels))\n",
    "\n",
    "print('NAIVE BAYES')\n",
    "print('=' * 55)\n",
    "print(classification_report(y_test, nb_pred, target_names=labels))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 7. Cross Validation"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "all_vec = vectorizer.transform(train_texts)\n",
    "\n",
    "lr_cv = cross_val_score(LogisticRegression(max_iter=1000), all_vec, train_labels, cv=5)\n",
    "nb_cv = cross_val_score(MultinomialNB(), all_vec, train_labels, cv=5)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 5))\n",
    "\n",
    "x = np.arange(5)\n",
    "width = 0.35\n",
    "\n",
    "ax.bar(x - width/2, lr_cv * 100, width, label='Logistic Regression', color='#C8F04A', edgecolor='none')\n",
    "ax.bar(x + width/2, nb_cv * 100, width, label='Naive Bayes', color='#42B4F5', edgecolor='none')\n",
    "\n",
    "ax.set_xlabel('Fold', color='#F0EEE8')\n",
    "ax.set_ylabel('Accuracy (%)', color='#F0EEE8')\n",
    "ax.set_title('5-Fold Cross Validation', color='#F0EEE8', fontweight='bold')\n",
    "ax.set_xticks(x)\n",
    "ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])\n",
    "ax.set_ylim(0, 115)\n",
    "ax.legend(facecolor='#16161A', edgecolor='#2A2A32', labelcolor='#F0EEE8')\n",
    "ax.grid(axis='y', alpha=0.2)\n",
    "\n",
    "ax.text(0.02, 0.95, f'LR Mean: {lr_cv.mean()*100:.1f}%',\n",
    "        transform=ax.transAxes, color='#C8F04A', fontsize=10)\n",
    "ax.text(0.02, 0.88, f'NB Mean: {nb_cv.mean()*100:.1f}%',\n",
    "        transform=ax.transAxes, color='#42B4F5', fontsize=10)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight', facecolor='#0D0D0F')\n",
    "plt.show()\n",
    "print(f'LR  → Mean: {lr_cv.mean()*100:.2f}% | Std: {lr_cv.std()*100:.2f}%')\n",
    "print(f'NB  → Mean: {nb_cv.mean()*100:.2f}% | Std: {nb_cv.std()*100:.2f}%')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 8. Word Clouds"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "category_texts = {\n",
    "    'Road'        : ' '.join([t for t,l in zip(train_texts,train_labels) if l=='road']),\n",
    "    'Garbage'     : ' '.join([t for t,l in zip(train_texts,train_labels) if l=='garbage']),\n",
    "    'Water'       : ' '.join([t for t,l in zip(train_texts,train_labels) if l=='water']),\n",
    "    'Electricity' : ' '.join([t for t,l in zip(train_texts,train_labels) if l=='electricity']),\n",
    "}\n",
    "\n",
    "wc_colors = ['#F5C842', '#6BCF7F', '#42B4F5', '#F57C42']\n",
    "\n",
    "fig, axes = plt.subplots(2, 2, figsize=(14, 8))\n",
    "fig.suptitle('Word Clouds — Top Keywords per Category',\n",
    "             color='#F0EEE8', fontsize=14, fontweight='bold')\n",
    "\n",
    "for ax, (cat, text), color in zip(axes.flatten(), category_texts.items(), wc_colors):\n",
    "    wc = WordCloud(\n",
    "        width=500, height=300,\n",
    "        background_color='#16161A',\n",
    "        colormap='YlOrRd',\n",
    "        max_words=30,\n",
    "        prefer_horizontal=0.9\n",
    "    ).generate(text)\n",
    "    ax.imshow(wc, interpolation='bilinear')\n",
    "    ax.set_title(cat, color=color, fontweight='bold', fontsize=12)\n",
    "    ax.axis('off')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('word_clouds.png', dpi=150, bbox_inches='tight', facecolor='#0D0D0F')\n",
    "plt.show()\n",
    "print('✅ Saved: word_clouds.png')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 9. TF-IDF Top Features"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "feature_names = vectorizer.get_feature_names_out()\n",
    "top_n = 8\n",
    "\n",
    "fig, axes = plt.subplots(2, 2, figsize=(14, 8))\n",
    "fig.suptitle('Top TF-IDF Features per Category',\n",
    "             color='#F0EEE8', fontsize=14, fontweight='bold')\n",
    "\n",
    "for ax, (cat, color) in zip(axes.flatten(),\n",
    "    zip(['road','garbage','water','electricity'], COLORS)):\n",
    "\n",
    "    class_idx = lr_model.classes_.tolist().index(cat)\n",
    "    coef = lr_model.coef_[class_idx]\n",
    "    top_idx = np.argsort(coef)[-top_n:]\n",
    "    top_features = [feature_names[i] for i in top_idx]\n",
    "    top_scores   = [coef[i] for i in top_idx]\n",
    "\n",
    "    ax.barh(top_features, top_scores, color=color, edgecolor='none')\n",
    "    ax.set_title(cat.capitalize(), color=color, fontweight='bold')\n",
    "    ax.set_xlabel('Coefficient', color='#F0EEE8')\n",
    "    ax.grid(axis='x', alpha=0.2)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('top_features.png', dpi=150, bbox_inches='tight', facecolor='#0D0D0F')\n",
    "plt.show()\n",
    "print('✅ Saved: top_features.png')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 10. Summary"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('=' * 50)\n",
    "print('  PROJECT SUMMARY')\n",
    "print('=' * 50)\n",
    "print(f'  Dataset Size      : {len(train_texts)} samples')\n",
    "print(f'  Categories        : 4 (Road, Garbage, Water, Electricity)')\n",
    "print(f'  Languages         : English + Hinglish')\n",
    "print(f'  Vectorizer        : TF-IDF (bigrams)')\n",
    "print(f'  Train/Test Split  : 80% / 20%')\n",
    "print('-' * 50)\n",
    "print(f'  Logistic Regression Accuracy : {lr_acc*100:.2f}%')\n",
    "print(f'  Naive Bayes Accuracy         : {nb_acc*100:.2f}%')\n",
    "print(f'  LR Cross Val Mean            : {lr_cv.mean()*100:.2f}%')\n",
    "print('-' * 50)\n",
    "winner = 'Logistic Regression' if lr_acc >= nb_acc else 'Naive Bayes'\n",
    "print(f'  Best Model : {winner}')\n",
    "print('=' * 50)\n",
    "print('  Graphs saved:')\n",
    "print('  → dataset_distribution.png')\n",
    "print('  → confusion_matrix.png')\n",
    "print('  → model_comparison.png')\n",
    "print('  → cross_validation.png')\n",
    "print('  → word_clouds.png')\n",
    "print('  → top_features.png')\n",
    "print('=' * 50)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}


NameError: name 'null' is not defined